# Проверочный набор mimo50: сравнение версий отправок (Colab A100)

Замер на 129 публичных задачах (fastapi, rich, requests) не предсказывает Kaggle: v16 решает там 51 против ~20 у v3,
а на Kaggle у обоих 0.15. Набор **mimo50** -- 50 фиксированных задач MiMo из 49 разных проектов (`eval/mimo50.json`),
на которых никто ничего не настраивал. Каждая версия запускается со своими промптом, лимитами, температурой и адаптером
из своего zip -- так же, как на Kaggle.

Проверка самого набора: на известных версиях порядок должен совпасть с Kaggle -- **v5 (0.20) выше v3 (0.15) и v16 (0.15)**.
Тогда набору можно доверять для новых версий.

Оговорки: (1) задачи отобраны из прогона с адаптером `pref_edit_v1` (14 решённых, 24 tests_failed, 12 no_patch), это
немного играет в пользу адаптера; (2) `thinking_budget` локально не соблюдается; (3) шум: при ~50 задачах разница
меньше ~4-5 задач может быть случайной, смотрите и на попарное сравнение (какие задачи решила одна версия и не решила другая).

Порядок: **0** версии, **1** карта, **2** код, **3** Drive и отправки, **4** модель, **5** vLLM со всеми адаптерами,
**6** сервер песочницы (MiMo), **7** прогоны (возобновляемые, результаты в Drive), **8** сравнение.

## 0. Версии (zip-отправки в `MyDrive/gemma_adapters/`)
Имя версии -- любое короткое; zip -- ровно то, что отправлялось на Kaggle. В третьем поле -- известный счёт (или None).

In [ ]:
VERSIONS = [
    # (имя, путь к zip внутри gemma_adapters, счёт Kaggle или None); идут по порядку: главное -- первым
    ("v16",      "submissions/teammate_v16.zip",                0.15),
    ("v16+pe16", "upload_pref_edit_v16/submission.zip",         None),   # v16 + pref_edit_v16 (colab_pref.ipynb)
    ("v5",       "submissions/v5.zip",                          0.20),   # v3 + pref_edit_v1
    ("v3",       "submissions/v3.zip",                          0.15),
]
WORKERS = 4        # не больше --max-sessions сервера

## 1. Карта

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)

## 2. Код и зависимости

In [ ]:
import os
if not os.path.exists("/content/gemma4-swe-rl"):
    !git clone -q https://github.com/aletonfdg/gemma4-swe-rl.git /content/gemma4-swe-rl
%cd /content/gemma4-swe-rl
!git pull -q
!git log --oneline | head -3
!pip install -q vllm openai pyyaml requests huggingface_hub kagglehub
import json
IDS = json.load(open("eval/mimo50.json", encoding="utf-8"))["ids"]
assert len(IDS) == 50 and len(set(IDS)) == 50
print("задач в наборе:", len(IDS))

## 3. Drive и отправки: промпт, лимиты, температура, адаптер каждой версии

In [ ]:
import re, shutil, zipfile, yaml
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/gemma_adapters"
OUT_ROOT = f"{DRIVE}/eval_mimo50"
os.makedirs(OUT_ROOT, exist_ok=True)

SPECS = {}
for name, zpath, score in VERSIONS:
    src = f"{DRIVE}/{zpath}"
    assert os.path.exists(src), f"нет {src}"
    d = f"/content/subs/{name}"
    shutil.rmtree(d, ignore_errors=True)
    zipfile.ZipFile(src).extractall(d)
    root = next(p for p, _, files in os.walk(d) if "agent.yaml" in files)
    agent = open(f"{root}/agent.yaml", encoding="utf-8").read()
    m = re.search(r"(?m)^adapter:\s*(\S+)", agent)
    adapter = m.group(1) if m else None
    ev = {}
    if os.path.exists(f"{root}/eval_config.yaml"):
        ev = (yaml.safe_load(open(f"{root}/eval_config.yaml")) or {}).get("evaluation", {})
    sampling = yaml.safe_load(open(f"{root}/configs/sampling.yaml")) if os.path.exists(f"{root}/configs/sampling.yaml") else {}
    if adapter:
        assert os.path.exists(f"{root}/adapters/{adapter}/adapter_model.safetensors"), f"{name}: нет файлов адаптера"
    SPECS[name] = {"root": root, "adapter": adapter, "score": score, "system": f"{root}/prompts/system.md",
                   "calls": ev.get("max_tool_calls"), "minutes": ev.get("max_time_minutes"),
                   "temperature": (sampling or {}).get("temperature", 0.2)}
    print(name, {k: v for k, v in SPECS[name].items() if k != "root"})

## 4. Базовая модель в `/content/model` (если уже есть, ничего не качает)

In [ ]:
import getpass
MODEL_DIR = "/content/model"
if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    try:
        from huggingface_hub import snapshot_download
        hf_token = getpass.getpass("HF токен (Enter, если не нужен): ").strip() or None
        snapshot_download("google/gemma-4-31b-it-qat-w4a16-ct", local_dir=MODEL_DIR, token=hf_token)
    except Exception as e:
        print("HF не получилось:", type(e).__name__, str(e)[:300])
        import kagglehub
        os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("KAGGLE_API_TOKEN (KGAT_...): ").strip()
        path = kagglehub.model_download("google/gemma-4/transformers/gemma-4-31b-it-qat-w4a16-ct")
        !cp -r "{path}"/. {MODEL_DIR}/
!du -sh {MODEL_DIR}

## 5. vLLM со всеми адаптерами версий (один запуск на все прогоны)

In [ ]:
import time, requests
BASE = "gemma-4-31b-it-qat-w4a16-ct"
loras = {s["adapter"]: f"{s['root']}/adapters/{s['adapter']}" for s in SPECS.values() if s["adapter"]}
cmd = ["vllm", "serve", MODEL_DIR, "--served-model-name", BASE,
       "--quantization", "compressed-tensors", "--dtype", "bfloat16",
       "--max-model-len", "32768", "--gpu-memory-utilization", "0.92", "--max-num-seqs", str(WORKERS),
       "--enable-auto-tool-choice", "--tool-call-parser", "gemma4", "--reasoning-parser", "gemma4",
       "--default-chat-template-kwargs", '{"enable_thinking": true}', "--port", "8000"]
if loras:
    cmd += ["--enable-lora", "--max-lora-rank", "64", "--max-loras", str(len(loras)), "--lora-modules"]
    cmd += [f"{k}={v}" for k, v in loras.items()]
proc = subprocess.Popen(cmd, stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
t0 = time.time()
while time.time() - t0 < 1500 and proc.poll() is None:
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=3).status_code == 200:
            print("vLLM готов за %.0f c" % (time.time() - t0)); break
    except requests.RequestException:
        pass
    time.sleep(5)
if proc.poll() is not None:
    print("vLLM упал:"); print(open("/content/vllm.log").read()[-3000:])
print([m["id"] for m in requests.get("http://127.0.0.1:8000/v1/models").json()["data"]])

## 6. Сервер песочницы на ноутбуке (задачи MiMo)
Docker Desktop запущен, ноутбук не засыпает. Без `--prune-images`: 50 образов скачаются один раз и пригодятся всем
версиям (проверьте, что на диске Docker есть ~100-150 ГБ: `docker system df`). Два окна PowerShell из папки проекта:
```
python -m sim.mimo_server --data C:\Users\vishn\Desktop\data\mimo_v2_6_code --max-sessions 4 --ttl 2400
cloudflared tunnel --url http://127.0.0.1:8765
```

In [ ]:
import getpass, urllib.request
os.environ["SWE_SANDBOX_URL"] = input("Адрес туннеля: ").strip().rstrip("/")
os.environ["SWE_SANDBOX_TOKEN"] = getpass.getpass("Токен сервера: ").strip()
req = urllib.request.Request(os.environ["SWE_SANDBOX_URL"] + "/tasks",
                             headers={"Authorization": "Bearer " + os.environ["SWE_SANDBOX_TOKEN"]})
server_ids = set(json.load(urllib.request.urlopen(req, timeout=30))["result"])
missing = [i for i in IDS if i not in server_ids]
assert not missing, f"сервер не знает {len(missing)} задач: запущен ли он с --data (MiMo)?"
print("сервер готов, задач:", len(server_ids))

## 7. Прогоны: каждая версия на 50 задачах (~1,5-2 ч на версию; первая дольше из-за скачивания образов)
Результаты сразу в Drive (`eval_mimo50/<версия>`); после обрыва повторите 2-6 и эту ячейку -- готовые эпизоды пропустятся.
Можно выполнить только часть версий: впишите их в `RUN_ONLY`.

In [ ]:
RUN_ONLY = None          # например ["v5"]; None -- все версии по порядку
for name, s in SPECS.items():
    if RUN_ONLY and name not in RUN_ONLY:
        continue
    budget = []
    if s["calls"]:
        budget += ["--tool-calls", str(s["calls"])]
    if s["minutes"]:
        budget += ["--time-minutes", str(s["minutes"])]
    run_cmd = " ".join(["python -m sim.run_mimo_baseline --model", s["adapter"] or BASE,
                        "--temperature", str(s["temperature"]), "--repeats 1 --workers", str(WORKERS),
                        "--thinking on --no-analyzer --clean-start --system-file", s["system"], *budget,
                        "--out", f"{OUT_ROOT}/{name}", "--ids", *IDS])
    print("=" * 30, name, "=" * 30); print(run_cmd)
    !{run_cmd}

## 8. Сравнение версий
Главный вопрос: совпадает ли порядок с Kaggle. Попарная таблица показывает, сколько задач версия-строка решила, а
версия-столбец нет: если разница держится на 1-2 задачах, это шум.

In [ ]:
def load(name):
    p = f"{OUT_ROOT}/{name}/results.jsonl"
    last = {}
    if os.path.exists(p):
        for l in open(p, encoding="utf-8"):
            if l.strip():
                r = json.loads(l); last[r["instance_id"]] = r
    return last

res = {name: load(name) for name in SPECS}
print(f"{'версия':<8}{'Kaggle':>8}{'решено':>9}{'из':>5}{'no_patch':>10}{'tests_failed':>14}")
for name, rows in res.items():
    reasons = [((r.get('result') or {}).get('reason') or ('tests_passed' if r['passed'] else '?')) for r in rows.values()]
    kag = SPECS[name]["score"]
    print(f"{name:<8}{kag if kag is not None else '-':>8}{sum(r['passed'] for r in rows.values()):>9}{len(rows):>5}"
          f"{reasons.count('no_patch'):>10}{reasons.count('tests_failed'):>14}")
names = list(res)
print("\nрешила строка, не решил столбец (только общие задачи):")
W = max(10, max(len(n) for n in names) + 2)
print(" " * W + "".join(f"{n:>{W}}" for n in names))
for a in names:
    line = f"{a:<{W}}"
    for b in names:
        common = set(res[a]) & set(res[b])
        line += f"{sum(res[a][t]['passed'] and not res[b][t]['passed'] for t in common):>{W}}" if a != b else f"{'-':>{W}}"
    print(line)